In [14]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zlib
from datetime import datetime
print('imported')


imported


In [15]:
# -- Ensure latest src/algorithm and test utilities are imported (with reload) --

import importlib

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import src.algorithm
import tests.test_tree_tn

importlib.reload(src.algorithm)
importlib.reload(tests.test_tree_tn)

from src.algorithm import (
    TensorNetwork,
    estimate_contraction,
)

from tests.test_tree_tn import (
    build_binary_ttn,
    exact_contract_ttn_tree,
)

print('functions loaded (reimported)')


functions loaded (reimported)


In [16]:
# compare AIS performance (logspace beta schedule) across:
#   - uniform1: varying jitter (spread around 1)
#   - diagexp:  varying alpha (decay rate)
#   - spikes:   varying spike_factor (diagonal dominance)
# fixed: A=200 (beta steps), B=40 (chains), C=200 (iters per step)

# fixed parameters
A, B, C = 200, 40, 200
n_trials = 10
EPS = 1e-30
base_seed = 42
dim = 3
depth = 4  # tree depth (2^depth = 16 leaves)

# step error metric: schedule-invariant log-ratio density error per unit β
#   density_abs: |log ρ̂_k - log ρ_k| / Δβ_k
#   density_rel: |û_k - u_k| / (|u_k| + ε), u_k := log ρ_k / Δβ_k
STEP_ERROR_METRIC = "density_abs"

def make_logspace_betas(A):
    """
    Generate logspace beta schedule with much finer spacing near β=1.
    Note: The last Δβ is ~1e-6, so density_abs at the very end can spike
    (because we divide by a tiny number). This is not a code bug — it's why
    the integrated summaries E_L1, E_L2 are the better schedule-invariant scalars.
    """
    betas = 1.0 - np.logspace(0, np.log10(1e-6), A)
    betas[0] = 0.0
    betas[-1] = 1.0
    return np.sort(betas)

betas = make_logspace_betas(A)
K = len(betas) - 1
beta_left = betas[:-1]
beta_right = betas[1:]
delta_beta = np.diff(betas)
beta_mid = 0.5 * (beta_left + beta_right)

# tn config for tree tensor networks
# each entry: (tensor_type, param_name, param_values, param_key)
tn_configs = {
    "uniform1": {
        "tensor_type": "uniform1",
        "param_name": "jitter",
        "param_values": [0.05, 0.1, 0.2, 0.4],
        "param_key": "jitter",
        "title": "Uniform Around 1",
    },
    "diagexp": {
        "tensor_type": "diagexp",
        "param_name": "α",
        "param_values": [0.2, 0.4, 0.6, 1.0],
        "param_key": "alpha",
        "title": "Exponential Diagonal Decay",
    },
    "spikes": {
        "tensor_type": "spikes",
        "param_name": "spike",
        "param_values": [2.0, 5.0, 10.0, 20.0],
        "param_key": "spike_factor",
        "title": "Diagonal Spikes",
    },
}

print(f"Tree TN config: depth={depth}, leaves={2**depth}, dim={dim}")
print(f"AIS config: A={A}, B={B}, C={C}, n_trials={n_trials}")


Tree TN config: depth=4, leaves=16, dim=3
AIS config: A=200, B=40, C=200, n_trials=10


In [17]:
# helpers
def _to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    if hasattr(x, "to_numpy"):
        return x.to_numpy()
    return np.asarray(x)

def stable_hash_int(s: str) -> int:
    """Stable hash function for reproducibility (replaces Python's non-deterministic hash())."""
    return zlib.crc32(s.encode("utf-8"))  # 0..2**32-1

def _seed_everything(s):
    """Seed random number generators, capping seed to 32-bit range for consistency."""
    s = int(s) % (2**32 - 1)
    np.random.seed(s)
    try:
        import cupy as cp
        cp.random.seed(s)
    except Exception:
        pass

def geom_mean_and_band(err_arr, eps=EPS, z=1.0):
    """Geometric mean and ±z*SE bands for 1D array."""
    loge = np.log(np.clip(err_arr, eps, None))
    mu = loge.mean()
    sd = loge.std(ddof=1)
    se = sd / np.sqrt(len(err_arr))
    g = np.exp(mu)
    lo = np.exp(mu - z * se)
    hi = np.exp(mu + z * se)
    return g, lo, hi

def _power_tensor_dict(tensors: dict, beta: float, tiny: float = 1e-30):
    """Raise all tensor entries to power beta."""
    out = {}
    for name, (arr, inds) in tensors.items():
        if beta == 0.0:
            out[name] = ((arr > 0).astype(float), inds)
        else:
            out[name] = (np.power(np.maximum(arr, tiny), beta), inds)
    return out

def exact_Z_over_betas_tree(G, tensors, betas, dim):
    """Compute exact Z(β) sequence for tree TN using message passing."""
    Zs = []
    for b in betas:
        tensors_b = _power_tensor_dict(tensors, b)
        Z = exact_contract_ttn_tree(G, tensors_b, dim=dim, root="I0_0")
        Zs.append(Z)
    return np.asarray(Zs, dtype=float)

print('helpers loaded')


helpers loaded


In [18]:
# run experiments
results = {}  # tn_type -> {param_val -> {"Z1_errs": [...], "step_errs": [...]}}

for tn_type, cfg in tn_configs.items():
    print(f"\n{'#' * 80}")
    print(f"# TN TYPE: {cfg['title']} ({tn_type})")
    print(f"{'#' * 80}")
    
    results[tn_type] = {}
    
    for param_val in cfg["param_values"]:
        print(f"\n{'=' * 60}")
        print(f"[{tn_type}] {cfg['param_name']} = {param_val}")
        print("=" * 60)
        
        # create Tree TN with this parameter
        tn_kwargs = {cfg["param_key"]: param_val}
        G, tensors = build_binary_ttn(
            depth=depth,
            dim=dim,
            tensor_type=cfg["tensor_type"],
            seed=base_seed,
            **tn_kwargs
        )
        tn = TensorNetwork(G, tensors)
        
        # compute exact Z(beta) for this TN
        print(f"  [info] computing exact Z(β) sequence...")
        t0 = time.time()
        Z_true_seq = exact_Z_over_betas_tree(G, tensors, betas, dim)
        Z_true_final = float(Z_true_seq[-1])
        true_ratios = Z_true_seq[1:] / Z_true_seq[:-1]
        
        # cheap assert: validate true_ratios
        if not np.all(np.isfinite(true_ratios)) or np.any(true_ratios <= 0):
            raise ValueError("true_ratios must be positive finite for log metric")
        
        print(f"  [info] exact Z(1) = {Z_true_final:.6e} (took {time.time()-t0:.1f}s)")
        
        all_Z1_errs = []
        all_step_errs = []
        
        for trial in range(n_trials):
            t0 = time.time()
            trial_seed = base_seed + 10000 * trial + stable_hash_int(tn_type) % 1000 + int(param_val * 100)
            _seed_everything(trial_seed)
            
            # run AIS
            Z_ests, logZ_trajs, weights_by_beta = estimate_contraction(
                tn, betas, iters=C, burns=max(0, min(C // 10, C - 1)),
                n_chains=B, verbose=False
            )
            
            # compute step error: schedule-invariant per-unit-β log-increment density
            w = _to_numpy(weights_by_beta)
            w = np.asarray(w, dtype=np.float64)
            
            # robust shape validation: enforce (K, B)
            if w.ndim != 2:
                raise ValueError(f"weights_by_beta should be 2D, got shape {w.shape}")
            
            K_expected = len(betas) - 1
            if w.shape[0] == K_expected:
                pass  # already (K, B)
            elif w.shape[1] == K_expected:
                w = w.T  # transpose from (B, K) to (K, B)
            else:
                raise ValueError(f"weights_by_beta shape {w.shape} incompatible with K={K_expected}")
            
            rho_hat = w.mean(axis=1)  # (K,)
            
            # cheap assert: validate rho_hat
            if not np.all(np.isfinite(rho_hat)):
                raise ValueError("rho_hat has non-finite values")
            if np.any(rho_hat <= 0):
                print("[warn] rho_hat has nonpositive entries; clipping will occur")
            log_rho_hat = np.log(np.clip(rho_hat, EPS, None))
            log_rho_true = np.log(np.clip(true_ratios, EPS, None))
            db = np.clip(delta_beta, 1e-15, None)
            u_hat = log_rho_hat / db
            u_true = log_rho_true / db
            step_err = np.abs(u_hat - u_true)
            if STEP_ERROR_METRIC == "density_rel":
                step_err = step_err / (np.abs(u_true) + 1e-12)
            all_step_errs.append(step_err)
            
            # schedule-invariant integrated scalars per trial (optional)
            # E_L1 ≈ ∑ Δβ_k e_k ; E_L2 ≈ (∑ Δβ_k e_k^2)^{1/2}
            EL1 = float(np.sum(delta_beta * step_err))
            EL2 = float(np.sqrt(np.sum(delta_beta * (step_err ** 2))))
            
            # Z(1) error
            Z_ests_np = _to_numpy(Z_ests)
            Z1_est = float(np.mean(Z_ests_np))
            Z1_log_err = abs(np.log(max(Z1_est, EPS)) - np.log(max(Z_true_final, EPS)))
            all_Z1_errs.append(Z1_log_err)
            
            dt = time.time() - t0
            print(f"    [trial {trial+1:02d}/{n_trials}] Z1_err={Z1_log_err:.3e}, time={dt:.1f}s")
        
        results[tn_type][param_val] = {
            "Z1_errs": np.array(all_Z1_errs),
            "step_errs": np.vstack(all_step_errs),  # (n_trials, K)
            # optional schedule-invariant integrated scalars (recomputed from step_errs when needed)
            "Z_true_final": Z_true_final,
        }
        
        # Z1_log_err is already in log space, so arithmetic mean is the natural aggregation
        mean_Z1 = float(np.mean(all_Z1_errs))
        print(f"  [summary] mean Z(1) error: {mean_Z1:.3e}")

print("\n[info] all experiments complete!")



################################################################################
# TN TYPE: Uniform Around 1 (uniform1)
################################################################################

[uniform1] jitter = 0.05
  [info] computing exact Z(β) sequence...
  [info] exact Z(1) = 2.104524e+14 (took 0.0s)
    [trial 01/10] Z1_err=7.603e-03, time=40.2s
    [trial 02/10] Z1_err=5.149e-03, time=55.9s
    [trial 03/10] Z1_err=1.035e-02, time=22.4s
    [trial 04/10] Z1_err=4.501e-03, time=17.7s
    [trial 05/10] Z1_err=4.863e-03, time=38.3s
    [trial 06/10] Z1_err=2.883e-03, time=19.2s
    [trial 07/10] Z1_err=7.493e-03, time=20.5s
    [trial 08/10] Z1_err=6.338e-03, time=19.5s
    [trial 09/10] Z1_err=3.452e-03, time=29.4s
    [trial 10/10] Z1_err=4.065e-03, time=24.1s
  [summary] mean Z(1) error: 5.670e-03

[uniform1] jitter = 0.1
  [info] computing exact Z(β) sequence...
  [info] exact Z(1) = 2.142532e+14 (took 0.0s)
    [trial 01/10] Z1_err=1.730e-02, time=24.6s
    [trial 02

KeyboardInterrupt: 

In [ ]:
# plotting
sns.set_theme(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Logspace β-Schedule: AIS Performance Across Tree TN Structures\n"
    f"(Binary TTN, depth={depth}, leaves={2**depth}, dim={dim}, A={A}, B={B}, C={C}, {n_trials} trials)",
    fontsize=13, fontweight="bold"
)

for ax_idx, (tn_type, cfg) in enumerate(tn_configs.items()):
    ax = axes[ax_idx]
    param_vals = cfg["param_values"]
    colors = sns.color_palette("viridis", n_colors=len(param_vals))
    
    for i, param_val in enumerate(param_vals):
        Z1_errs = results[tn_type][param_val]["Z1_errs"]
        g, lo, hi = geom_mean_and_band(Z1_errs)
        
        # bar plot with error bars
        ax.bar(
            i, g, width=0.7, color=colors[i], alpha=0.8,
            yerr=[[g - lo], [hi - g]], capsize=5,
            label=f"{cfg['param_name']}={param_val}"
        )
    
    ax.set_yscale("log")
    ax.set_xticks(range(len(param_vals)))
    ax.set_xticklabels([str(v) for v in param_vals])
    ax.set_xlabel(cfg["param_name"])
    ax.set_ylabel(r"$Z(1)$ Log Error: $|\log\hat{Z} - \log Z|$")
    ax.set_title(cfg["title"])
    ax.grid(True, which="both", alpha=0.3, axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

# save the figure
plots_dir = os.path.join(os.path.dirname(os.getcwd()), "plots")
os.makedirs(plots_dir, exist_ok=True)
fig_path = os.path.join(plots_dir, "ais_tree_tn_comparison.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"[done] saved plot -> {fig_path}")


In [ ]:
# additional plot: step-ratio error curves
if STEP_ERROR_METRIC == "density_abs":
    metric_label = r"Abs density err: $\frac{|\log\hat{\rho}_k-\log\rho_k|}{\Delta\beta_k}$"
    metric_name = "Density-Abs"
else:  # density_rel
    metric_label = r"Rel density err: $\frac{|\hat{u}_k-u_k|}{|u_k|+\epsilon}$"
    metric_name = "Density-Rel"

fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))
fig2.suptitle(
    f"Logspace β-Schedule: Per-Step Error by Tree TN Structure ({metric_name})\n"
    f"(Binary TTN, depth={depth}, leaves={2**depth}, dim={dim}, A={A}, B={B}, C={C}, {n_trials} trials)",
    fontsize=13, fontweight="bold"
)

beta_x = beta_mid

for ax_idx, (tn_type, cfg) in enumerate(tn_configs.items()):
    ax = axes2[ax_idx]
    param_vals = cfg["param_values"]
    colors = sns.color_palette("viridis", n_colors=len(param_vals))
    
    for i, param_val in enumerate(param_vals):
        step_errs = results[tn_type][param_val]["step_errs"]  # (n_trials, K)
        
        # geometric mean and bands across trials at each beta
        gmeans, los, his = [], [], []
        for k in range(K):
            g, lo, hi = geom_mean_and_band(step_errs[:, k])
            gmeans.append(g)
            los.append(lo)
            his.append(hi)
        
        ax.semilogy(
            beta_x, gmeans, marker="o", ms=1.5, lw=1.2, color=colors[i],
            label=f"{cfg['param_name']}={param_val}"
        )
        ax.fill_between(beta_x, los, his, color=colors[i], alpha=0.15)
    
    ax.set_xlabel(r"$\beta_{k+1/2}$")
    ax.set_ylabel(metric_label)
    ax.set_title(cfg["title"])
    ax.legend(loc="best", fontsize="small")
    ax.grid(True, which="both", alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

fig2_path = os.path.join(plots_dir, "ais_tree_tn_step_errors.png")
fig2.savefig(fig2_path, dpi=300, bbox_inches="tight")
print(f"[done] saved plot -> {fig2_path}")


In [ ]:
# save data
data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")
os.makedirs(data_dir, exist_ok=True)

rows = []
for tn_type, cfg in tn_configs.items():
    for param_val in cfg["param_values"]:
        res = results[tn_type][param_val]
        for t in range(n_trials):
            for k in range(K):
                step_err = float(res["step_errs"][t, k])
                # integrated scalars (constant across k for a given trial)
                EL1 = float(np.sum(delta_beta * res["step_errs"][t, :]))
                EL2 = float(np.sqrt(np.sum(delta_beta * (res["step_errs"][t, :] ** 2))))
                rows.append({
                    "tn_type": tn_type,
                    "param_name": cfg["param_name"],
                    "param_value": param_val,
                    "depth": depth,
                    "dim": dim,
                    "A": A, "B": B, "C": C,
                    "trial": t,
                    "beta_idx": k,
                    "beta_mid": float(beta_mid[k]),
                    "beta_left": float(beta_left[k]),
                    "beta_right": float(beta_right[k]),
                    "delta_beta": float(delta_beta[k]),
                    "step_error": step_err,
                    "step_error_metric": STEP_ERROR_METRIC,
                    "E_L1": EL1,
                    "E_L2": EL2,
                    "Z1_log_error": float(res["Z1_errs"][t]),
                    "Z_true_final": res["Z_true_final"],
                })

df_out = pd.DataFrame(rows)
csv_path = os.path.join(data_dir, "ais_tree_tn_comparison.csv")
df_out.to_csv(csv_path, index=False)
print(f"[done] saved data -> {csv_path}")
